# 03 — Feature Engineering

In this notebook we construct **predictive features** from the processed data created in the previous step.

At this stage the data consists of two separate datasets:

- **LOB data** containing order book snapshots with bid and ask prices and volumes at multiple depth levels.
- **Trade data** containing executed trades over time.

The goal of this notebook is to transform these raw structures into **useful market microstructure variables** that describe the state of the market. These features will capture aspects such as liquidity, order flow, price pressure, momentum, and short-term volatility.

The resulting features will later be combined into a **single modelling dataset** 

In [13]:
import pandas as pd
import numpy as np
from binance.client import Client
import time
import black
from pympler import asizeof
import math
import matplotlib.pyplot as plt
import glob

pd.set_option("display.max_columns", None)

e:\anaconda3\envs\lob-alpha\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.0.1)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(


# load in interim dataset

In [146]:
files_trades = glob.glob(
    "E:\\Quant_Projects\\microstructure-alpha-engine\\microstructure-alpha-engine\\data\\interim\\trades*"
)
files_lob = glob.glob(
    "E:\\Quant_Projects\\microstructure-alpha-engine\\microstructure-alpha-engine\\data\\interim\\lob_*"
)
lob_df = pd.concat([pd.read_parquet(f) for f in files_lob])
trade_df = pd.concat([pd.read_parquet(f) for f in files_trades])

trade_df

,id,price,qty,quoteQty,time,isBuyerMaker,isBestMatch,timestamp
1000,6086761823,69913.84,0.00008,5.593107,1773122650687,False,True,1773122649927
1001,6086761822,69913.80,0.00008,5.593104,1773122650687,False,True,1773122649927
1002,6086761826,69913.85,0.00008,5.593108,1773122650687,False,True,1773122649927
1003,6086761827,69913.85,0.00008,5.593108,1773122650687,False,True,1773122649927
1004,6086761824,69913.85,0.00008,5.593108,1773122650687,False,True,1773122649927
...,...,...,...,...,...,...,...,...
254804,6087019438,70344.75,0.09021,6345.799898,1773128553949,False,True,1773128552519
254805,6087019439,70344.74,0.00571,401.668465,1773128553955,True,True,1773128552519
254806,6087019440,70344.74,0.00106,74.565424,1773128554030,True,True,1773128553986
254807,6087019441,70344.75,0.00025,17.586188,1773128554358,False,True,1773128553986


# Creating the variables for Limit order books

In [147]:
EPS = 1e-9

In [148]:
def create_lob_price_features(df):
    df["mid_price"] = (df["lob_bids_price_1"] + df["lob_asks_price_1"]) / 2
    df["spread"] = df["lob_asks_price_1"] - df["lob_bids_price_1"]
    df["rel_spread"] = df["spread"] / (df["mid_price"] + EPS)
    return df


def lob_volume_features(df):
    df["liquidity"] = df["lob_bids_volume_1"] + df["lob_asks_volume_1"]
    df["total_bid_volume_10"] = df[[f"lob_bids_volume_{i}" for i in range(1, 11)]].sum(
        axis=1
    )
    df["total_ask_volume_10"] = df[[f"lob_asks_volume_{i}" for i in range(1, 11)]].sum(
        axis=1
    )
    df["total_book_volume"] = df["total_ask_volume_10"] + df["total_bid_volume_10"]
    df["max_bid_ask_vol_ratio"] = df["lob_bids_volume_1"] / (
        df["lob_asks_volume_1"] + EPS
    )
    return df


def lob_pressure_features(df):

    df["imbalance_1"] = (df["lob_bids_volume_1"] - df["lob_asks_volume_1"]) / (
        df["lob_bids_volume_1"] + df["lob_asks_volume_1"] + EPS
    )

    bid_vol5 = df[[f"lob_bids_volume_{i}" for i in range(1, 6)]].sum(axis=1)
    ask_vol5 = df[[f"lob_asks_volume_{i}" for i in range(1, 6)]].sum(axis=1)

    df["imbalance_5"] = (bid_vol5 - ask_vol5) / (bid_vol5 + ask_vol5 + EPS)

    bid10 = df[[f"lob_bids_volume_{i}" for i in range(1, 11)]].sum(axis=1)
    ask10 = df[[f"lob_asks_volume_{i}" for i in range(1, 11)]].sum(axis=1)

    df["imbalance_10"] = (bid10 - ask10) / (bid10 + ask10 + EPS)

    df["microprice"] = (
        df["lob_bids_volume_1"] * df["lob_asks_price_1"]
        + df["lob_asks_volume_1"] * df["lob_bids_price_1"]
    ) / (df["liquidity"] + EPS)

    df["microprice_change"] = df["microprice"].diff()
    df["mid_minus_micro"] = df["mid_price"] - df["microprice"]

    numerator10 = sum(
        df[f"lob_bids_volume_{i}"] * df[f"lob_asks_price_{i}"]
        + df[f"lob_asks_volume_{i}"] * df[f"lob_bids_price_{i}"]
        for i in range(1, 11)
    )

    denominator10 = sum(
        df[f"lob_bids_volume_{i}"] + df[f"lob_asks_volume_{i}"] for i in range(1, 11)
    )
    df["microprice_weighted_10"] = numerator10 / (denominator10 + EPS)
    return df


def lob_returns_and_momentum_features(df):
    df["return_1"] = df["mid_price"].pct_change(1)
    df["return_5"] = df["mid_price"].pct_change(5)

    df["log_return_1"] = np.log(df["mid_price"]).diff(1)
    df["log_return_2"] = np.log(df["mid_price"]).diff(2)
    df["log_return_3"] = np.log(df["mid_price"]).diff(3)
    df["log_return_5"] = np.log(df["mid_price"]).diff(5)
    df["log_return_20"] = np.log(df["mid_price"]).diff(20)
    # momentum based return

    df["momentum_5_log_return_1"] = df["log_return_1"].rolling(5).mean()
    df["momentum_20_log_return_1"] = df["log_return_1"].rolling(20).mean()

    return df


def lob_volatility_features(df):
    df["vol_5"] = df["log_return_1"].rolling(5).std()
    df["vol_20"] = df["log_return_1"].rolling(20).std()

    df["realized_vol_5"] = np.sqrt((df["log_return_1"] ** 2).rolling(5).sum())
    df["realized_vol_20"] = np.sqrt((df["log_return_1"] ** 2).rolling(20).sum())
    return df


def lob_target_feature(df):
    df["mid_price_change"] = df["mid_price"].shift(-1) - df["mid_price"]
    df["mid_price_change_sign"] = np.sign(df["mid_price_change"])
    return df

## Pipeline

In [149]:
def build_lob_feature_pipeline(df):

    df = create_lob_price_features(df)

    df = lob_volume_features(df)

    df = lob_pressure_features(df)

    df = lob_returns_and_momentum_features(df)

    df = lob_volatility_features(df)

    df = lob_target_feature(df)

    return df

# Create trade features

In [150]:
def trade_base_features(trade_df):

    df = trade_df.copy()

    df["buy_trade"] = (df["isBuyerMaker"] == False).astype(int)
    df["sell_trade"] = (df["isBuyerMaker"] == True).astype(int)

    df["buy_qty"] = df["qty"].where(df["isBuyerMaker"] == False, 0)
    df["sell_qty"] = df["qty"].where(df["isBuyerMaker"] == True, 0)

    df["price_qty"] = df["price"] * df["qty"]

    agg = df.groupby("timestamp").agg(
        trade_count=("qty", "count"),
        buy_count=("buy_trade", "sum"),
        sell_count=("sell_trade", "sum"),
        total_trade_volume=("qty", "sum"),
        buy_volume=("buy_qty", "sum"),
        sell_volume=("sell_qty", "sum"),
        avg_trade_size=("qty", "mean"),
        max_trade_size=("qty", "max"),
        min_trade_size=("qty", "min"),
        std_trade_size=("qty", "std"),
        vwap_num=("price_qty", "sum"),
    )

    agg["vwap"] = agg["vwap_num"] / (agg["total_trade_volume"] + EPS)

    agg = agg.drop(columns="vwap_num")

    return agg.reset_index()


def trade_pressure_features(df):
    df["trade_volume_imbalance"] = (df["buy_volume"] - df["sell_volume"]) / (
        df["buy_volume"] + df["sell_volume"] + EPS
    )
    df["trade_flow_ratio"] = df["buy_volume"] / (df["sell_volume"] + EPS)

    return df


def trade_change_features(df):

    df["trade_volume_change"] = df["total_trade_volume"].diff()

    df["trade_count_change"] = df["trade_count"].diff()

    return df


def trade_lag_features(df):

    df["lag_trade_volume_imbalance_1"] = df["trade_volume_imbalance"].shift(1)
    df["lag_trade_volume_imbalance_2"] = df["trade_volume_imbalance"].shift(2)
    df["lag_trade_volume_imbalance_3"] = df["trade_volume_imbalance"].shift(3)
    df["lag_trade_volume_imbalance_5"] = df["trade_volume_imbalance"].shift(5)

    return df

## Pileline

In [151]:
def build_trade_feature_pipeline(df):
    df = trade_base_features(df)

    df = trade_pressure_features(df)

    df = trade_change_features(df)

    df = trade_lag_features(df)

    return df

In [152]:
def merge_lob_and_trade(trade_features_df, lob_features_df):
    final_df = lob_features_df.merge(trade_features_df, on="timestamp", how="left")
    return final_df

# Building features and Final dataset and saving

In [ ]:
lob_with_features = build_lob_feature_pipeline(lob_df)
trade_with_features = build_trade_feature_pipeline(trade_df)

final_dataset = merge_lob_and_trade(trade_with_features, lob_with_features)

final_dataset.dropna(inplace=True)

PATH = "E:\\Quant_Projects\\microstructure-alpha-engine\\microstructure-alpha-engine\\data\\processed\\final_dataset.parquet"

final_dataset.to_parquet(
    PATH,
    compression="snappy",
)

# Batch friendly Version for bigger datasets

In [ ]:
from microstructure_alpha.features.lob_features import build_lob_feature_pipeline
from microstructure_alpha.features.trade_features import build_trade_feature_pipeline
from microstructure_alpha.features.merge_features import merge_lob_and_trade
from microstructure_alpha.utils.logger import setup_logger
import time
import glob
from pathlib import Path
import gc
from tqdm import tqdm

In [2]:
logger = setup_logger(
    "E:\\Quant_Projects\\microstructure-alpha-engine\\microstructure-alpha-engine\\logs\\feature_engineering_notebook.log"
)

In [16]:
def run_batch_feature_engineering_pipeline(
    lob_path: str, trade_path: str, save_path: str
):
    files_trade = sorted(glob.glob(trade_path + "trade_interim_*"))
    files_lob = sorted(glob.glob(lob_path + "lob_interim_*"))

    logger.info("Starting batch feature pipeline")
    logger.info(f"Found {len(files_trade)} trade files")
    logger.info(f"Found {len(files_lob)} lob files")

    total_batches = min(len(files_trade), len(files_lob))
    logger.info(f"Processing {total_batches} batches")

    for i, (trade_file, lob_file) in tqdm(
        enumerate(zip(files_trade, files_lob)),
        total=total_batches,
        desc="Processing batches",
    ):

        lob_df = pd.read_parquet(lob_file)
        trade_df = pd.read_parquet(trade_file)

        lob_with_features = build_lob_feature_pipeline(lob_df)
        trade_with_features = build_trade_feature_pipeline(trade_df)

        final_dataset = merge_lob_and_trade(trade_with_features, lob_with_features)

        final_dataset.to_parquet(save_path + f"final_dataset_{i}", compression="snappy")
    logger.info("Batch feature pipeline completed successfully")
    return

In [ ]:
trade_path = "E:\\Quant_Projects\\microstructure-alpha-engine\\microstructure-alpha-engine\\data\\interim\\"

lob_path = "E:\\Quant_Projects\\microstructure-alpha-engine\\microstructure-alpha-engine\\data\\interim\\"

save_path = "E:\\Quant_Projects\\microstructure-alpha-engine\\microstructure-alpha-engine\\data\\processed\\"

RUN_PIPELINE = False
if RUN_PIPELINE:
    run_batch_feature_engineering_pipeline(lob_path, trade_path, save_path)

2026-03-14 15:24:28 | INFO | Starting batch feature pipeline
2026-03-14 15:24:28 | INFO | Found 5 trade files
2026-03-14 15:24:28 | INFO | Found 5 lob files
2026-03-14 15:24:28 | INFO | Processing 5 batches


Processing batches: 100%|██████████| 5/5 [00:00<00:00,  6.48it/s]

2026-03-14 15:24:29 | INFO | Batch feature pipeline completed successfully
